# Tabu Search — Theory & Mathematical Foundations

> **Note:** This notebook covers **theory only** (all markdown, no executable code).  
> For the full Python implementation, experiments, and visualisations, see [`notebook_implementation.ipynb`](notebook_implementation.ipynb) in the same directory.

---

## 1. Definition

### Origin

Tabu Search (TS) was introduced by **Fred Glover** in two landmark papers:

- Glover, F. (1986). *Future paths for integer programming and links to artificial intelligence.* Computers & Operations Research, 13(5), 533–549.
- Glover, F. (1989). *Tabu search — Part I.* ORSA Journal on Computing, 1(3), 190–206.

The term *tabu* (from the Polynesian word for "forbidden") reflects the algorithm's central mechanism: a **memory structure** that forbids revisiting recently explored solutions.

### Category

Tabu Search belongs to the family of **metaheuristics** — high-level, problem-independent strategies that guide an underlying local search to explore the solution space more effectively than naive methods. More specifically, TS is a **local search with memory**: it extends classical neighbourhood-based local search by maintaining an explicit record of recent moves to prevent cycling and to encourage exploration.

### Key Concepts

| Concept | Description |
|---|---|
| **Tabu list** $T$ | A short-term memory structure (typically a FIFO queue) that records recently performed moves or visited solutions and declares them *forbidden* for a number of iterations called the **tabu tenure** $\theta$. |
| **Aspiration criterion** $A$ | An override rule that allows a tabu move to be accepted despite being forbidden, usually when it leads to a solution **better than the best known** so far. |
| **Short-term memory** | The tabu list itself — prevents immediate cycling over the last $\theta$ moves. |
| **Long-term memory** | Optional structures (frequency-based or recency-based) used for **intensification** (exploiting promising regions) and **diversification** (forcing exploration of unvisited regions). |

### Why Tabu Search Improves on Plain Local Search

Classical local search (e.g., steepest descent) accepts only **improving moves** — it stops as soon as no neighbour has a lower cost. This makes it prone to getting **trapped in local optima**, which may be arbitrarily far from the global optimum on combinatorial landscapes like TSP.

Tabu Search addresses this by **allowing non-improving (uphill) moves**: at each iteration the best available move is accepted, even if it worsens the objective, as long as it is not tabu (or passes the aspiration criterion). The tabu list prevents the algorithm from immediately reversing such a move and cycling back, thereby forcing exploration of new regions of the solution space. This mechanism allows TS to **escape local optima** and continue searching, while the best-solution memory ensures that improvements are never lost.

---

## 2. Formal Description

### Problem Setting — TSP

Let $G = (V, E)$ be a complete undirected graph where $V = \{1, 2, \ldots, n\}$ is the set of cities and $d_{ij} \geq 0$ is the Euclidean distance between cities $i$ and $j$. The **Travelling Salesman Problem** seeks a Hamiltonian tour (a permutation $\pi$ of $V$) that minimises the total length:

$$
f(\pi) = \sum_{k=1}^{n} d_{\pi(k),\, \pi(k \bmod n \,+\, 1)}
$$

### Solution Space

The **solution space** is:

$$
\mathcal{S} = \{ \pi \in \mathfrak{S}_n \mid \pi \text{ is a Hamiltonian permutation of } V \}
$$

where $|\mathcal{S}| = (n-1)!/2$ for the symmetric TSP.

### Neighbourhood Structure — 2-opt

Given a current tour $s \in \mathcal{S}$, the **2-opt neighbourhood** $N(s)$ is defined by all tours obtainable by removing two non-adjacent edges $(i, i+1)$ and $(j, j+1)$ and reconnecting the four endpoints in the only alternative way:

$$
N(s) = \{ s' \in \mathcal{S} \mid s' = \text{2-opt}(s, i, j),\; 1 \leq i < j \leq n,\; j \neq i+1 \}
$$

The **cost delta** of a 2-opt move $(i, j)$ can be computed in $O(1)$ using the pre-computed distance matrix:

$$
\Delta f(s, i, j) = \bigl(d_{s(i),\, s(j)} + d_{s(i+1),\, s(j+1)}\bigr) - \bigl(d_{s(i),\, s(i+1)} + d_{s(j),\, s(j+1)}\bigr)
$$

A move is **improving** when $\Delta f < 0$.

### Tabu List

The **tabu list** $T$ is a FIFO queue of bounded capacity $\theta$ (the *tabu tenure*) that stores recently performed moves. A move $m = (i, j)$ is said to be **tabu** if $m \in T$. Formally:

$$
T \subset \mathcal{M},\quad |T| \leq \theta
$$

where $\mathcal{M}$ is the set of all possible moves.

### Aspiration Criterion

The **aspiration criterion** $A(s')$ is a Boolean predicate that overrides the tabu status of a move when accepting it is beneficial enough. The standard criterion is:

$$
A(s') = \mathbf{1}\bigl[f(s') < f(s^*) \bigr]
$$

where $s^*$ is the best solution found so far. In other words, a tabu move is *allowed* if it leads to a **global improvement** over the best known solution.

### Move Selection Rule

At each iteration, the algorithm selects the best move from the neighbourhood that is either not tabu or satisfies the aspiration criterion:

$$
s^{\text{next}} = \arg\min_{s' \in N(s)} \bigl\{ f(s') \;\mid\; \text{move}(s \to s') \notin T \;\lor\; A(s') \bigr\}
$$

### Tabu List Update

After each move $m^* = (i, j)$ is selected:

1. Append $m^*$ to $T$.
2. If $|T| > \theta$, evict the **oldest** entry (FIFO discipline).
3. Update $s^*$ if $f(s^{\text{next}}) < f(s^*)$.

This ensures that $m^*$ is forbidden for the next $\theta$ iterations, preventing immediate reversal.

---

## 3. Architecture / Algorithm Steps

### Full Algorithm Pseudocode

```
Algorithm: Tabu Search for TSP
Input : distance matrix D[1..n][1..n], max_iter, tabu_tenure θ, patience
Output: best tour s* and its cost f(s*)

1.  s      ← NearestNeighbour(D)        // greedy initialisation
2.  s*     ← s
3.  T      ← empty FIFO queue (capacity θ)
4.  no_improve ← 0

5.  FOR iter = 1 TO max_iter DO

6.      best_delta ← +∞
7.      best_move  ← None

8.      // --- 2-opt neighbourhood evaluation ---
9.      FOR i = 1 TO n-2 DO
10.         FOR j = i+2 TO n DO
11.             delta ← Δf(s, i, j)           // O(1) with distance matrix
12.             move  ← (i, j)
13.             IF (move ∉ T) OR (f(s) + delta < f(s*)) THEN  // not tabu OR aspiration
14.                 IF delta < best_delta THEN
15.                     best_delta ← delta
16.                     best_move  ← move
17.                 END IF
18.             END IF
19.         END FOR
20.     END FOR

21.     IF best_move = None THEN BREAK   // all moves tabu (rare edge case)

22.     // --- Apply best move ---
23.     s ← Apply2opt(s, best_move)

24.     // --- Update tabu list ---
25.     Enqueue(T, best_move)
26.     IF |T| > θ THEN Dequeue(T)       // evict oldest

27.     // --- Update best solution ---
28.     IF f(s) < f(s*) THEN
29.         s*        ← s
30.         no_improve ← 0
31.     ELSE
32.         no_improve ← no_improve + 1
33.     END IF

34.     IF no_improve ≥ patience THEN BREAK  // early stopping

35. END FOR

36. RETURN s*, f(s*)
```

### Control Flow Diagram

```
┌──────────────────────────────────────────────────────────┐
│               TABU SEARCH — CONTROL CYCLE                │
└──────────────────────────────────────────────────────────┘

  [START]
      │
      ▼
  ┌─────────────────────┐
  │  Initialise with    │
  │  nearest-neighbour  │
  └────────┬────────────┘
           │
           ▼
  ┌─────────────────────┐
  │  Generate 2-opt     │◄─────────────────────────────┐
  │  neighbourhood N(s) │                              │
  └────────┬────────────┘                              │
           │                                           │
           ▼                                           │
  ┌─────────────────────┐                              │
  │  For each s' ∈ N(s) │                              │
  │  check tabu status  │                              │
  │  & aspiration crit. │                              │
  └────────┬────────────┘                              │
           │                                           │
           ▼                                           │
  ┌─────────────────────┐                              │
  │  Select best        │                              │
  │  admissible move m* │                              │
  └────────┬────────────┘                              │
           │                                           │
           ▼                                           │
  ┌─────────────────────┐    ┌──────────────────────┐  │
  │  Apply move:        │    │  Update tabu list T: │  │
  │  s ← Apply2opt(s,m*)│───►│  add m*, evict if    │  │
  └─────────────────────┘    │  |T| > θ (FIFO)      │  │
                             └──────────┬───────────┘  │
                                        │               │
                                        ▼               │
                             ┌──────────────────────┐   │
                             │  f(s) < f(s*)?       │   │
                             │  YES → s* ← s        │   │
                             └──────────┬───────────┘   │
                                        │               │
                                        ▼               │
                             ┌──────────────────────┐   │
                             │  Stopping criterion  │   │
                             │  met? (max_iter or   │   │
                             │  patience exceeded)  │   │
                             │  NO ─────────────────────┘
                             │  YES
                             └──────────┬───────────┘
                                        │
                                        ▼
                                    [RETURN s*]
```

### Notes on Initialisation

The **nearest-neighbour heuristic** constructs an initial tour greedily: starting from city 1, it repeatedly visits the closest unvisited city. This produces a solution typically **20–25% above optimum** on random instances, giving TS a reasonable starting point. Alternatively, a random tour may be used when diversity is preferred over a warm start.

---

## 4. Complexity Analysis

### Per-Iteration Cost

The inner double loop over all 2-opt pairs $(i, j)$ with $1 \leq i < j \leq n$ evaluates exactly

$$
\binom{n}{2} - (n-1) = \frac{n(n-1)}{2} - (n-1) = \frac{(n-1)(n-2)}{2} = O(n^2)
$$

candidate moves (excluding adjacent pairs). Each evaluation requires:

- **$O(1)$ delta computation** — the cost change $\Delta f(s, i, j)$ involves only four distance lookups in the pre-computed matrix $D$:

$$
\Delta f = d_{s(i),\,s(j)} + d_{s(i+1),\,s(j+1)} - d_{s(i),\,s(i+1)} - d_{s(j),\,s(j+1)}
$$

- **$O(\theta)$ tabu check** — checking membership in $T$; using a hash set this reduces to $O(1)$ amortised.

Therefore the per-iteration cost is $\Theta(n^2)$.

### Applying a Move

A 2-opt reversal of the segment $s(i+1), \ldots, s(j)$ costs $O(n)$ in the worst case (reversing up to $n/2$ elements). For large $n$, this can be reduced to $O(\sqrt{n})$ amortised using **segment-tree** or **splay-tree** tour representations, but the standard array implementation is $O(n)$.

### Total Time Complexity

For `max_iter` iterations:

$$
T_{\text{total}} = O(\text{max\_iter} \times n^2)
$$

In practice, early stopping via a **patience** counter means the algorithm rarely runs all `max_iter` iterations on easy instances.

### Space Complexity

| Structure | Space |
|---|---|
| Distance matrix $D$ | $O(n^2)$ |
| Current tour $s$ | $O(n)$ |
| Best tour $s^*$ | $O(n)$ |
| Tabu list $T$ | $O(\theta)$ |
| **Total** | $O(n^2 + \theta)$ |

The $O(n^2)$ distance matrix dominates for large $n$. For $n = 1000$, this is approximately $8 \times 10^6$ floats ($\approx 64$ MB in float64), which is manageable on modern hardware.

### Practical Scalability

The $O(n^2)$ per-iteration cost means that doubling $n$ quadruples the runtime. Empirically:

- $n = 100$: milliseconds per iteration, total run in seconds.
- $n = 500$: tens of milliseconds per iteration, total run in minutes.
- $n = 1000$: hundreds of milliseconds per iteration; TS becomes slow without additional techniques (candidate lists, etc.).

---

## 5. Strengths and Limitations

| | Strengths | Limitations |
|---|---|---|
| **Optimality escape** | Escapes local optima by allowing non-improving moves, unlike plain 2-opt or greedy descent. | No guarantee of reaching the global optimum; quality depends on landscape and parameter settings. |
| **Parameter sensitivity** | Fewer critical parameters than genetic algorithms or simulated annealing (no temperature schedule). Tabu tenure $\theta$ is the main knob. | **Sensitive to tabu tenure $\theta$**: too small causes cycling; too large over-restricts the search (see Section 10). |
| **Reproducibility** | Fully deterministic given the initial solution; results are reproducible with a fixed random seed (if using random restarts). | If diversification via random restarts is added, reproducibility requires careful seed management. |
| **Practical performance** | Consistently produces solutions 5–15% above optimum on medium TSP instances ($n = 50$–$500$), well ahead of nearest-neighbour. | **Scalability**: $O(n^2)$ per iteration limits use beyond $n \approx 1000$ without advanced data structures or candidate lists. |
| **Simplicity** | Conceptually simple, easy to implement and debug. No probabilistic acceptance or complex operators needed for a basic version. | Memory structure adds bookkeeping overhead absent from plain local search. |
| **Flexibility** | Easily extended with long-term memory, candidate lists, and problem-specific moves. | Problem-specific tuning may be required to match the performance of specialised solvers (e.g., LKH). |

---

## 6. Use Case Explanation

### When to Use Tabu Search for TSP

**Recommended scenarios:**

- **Medium-scale TSP instances ($n = 50$–$500$):** Tabu Search produces high-quality solutions (typically within 5–15% of optimum) in a reasonable time budget. This is the sweet spot where TS is both fast enough and powerful enough to justify its complexity over simple heuristics.

- **When you need significantly better than nearest-neighbour:** NN heuristics are fast but typically 20–25% above optimum. TS reliably closes a large portion of that gap without the implementation complexity of Lin–Kernighan (LKH) or exact solvers.

- **As a strong baseline for deep learning comparison:** In research contexts comparing classical and learned solvers (e.g., Pointer Networks, Graph Neural Networks), TS provides a deterministic, well-understood, reproducible baseline that does not require labelled training data.

- **When training data is unavailable:** Unlike GNN-based solvers which require large datasets of solved instances, TS can be applied to any instance immediately.

- **When computational budget is moderate:** A TS run on $n = 100$ with 500 iterations completes in under a second on a standard CPU. This makes it suitable for real-time or near-real-time applications at this scale.

### When NOT to Use Tabu Search

- **Very large instances ($n > 5000$):** The $O(n^2)$ per-iteration cost becomes prohibitive. At $n = 5000$, a single iteration evaluates ~12.5 million 2-opt pairs. Specialised approaches (LKH with candidate lists, population-based methods, or approximation algorithms) are more appropriate.

- **When provably optimal solutions are required:** TS is a heuristic with no optimality certificate. If the application demands a proof of optimality (e.g., critical logistics planning with tight cost constraints), use exact methods: branch-and-bound, branch-and-cut, or the Concorde TSP solver.

- **When inference speed at deployment is the bottleneck:** In a machine learning pipeline where thousands of TSP instances must be solved per second at inference time, a trained GNN or Pointer Network is orders of magnitude faster than TS, even though TS may find better individual solutions.

---

## 10. Experimental Analysis

### Effect of Tabu Tenure $\theta$

The tabu tenure $\theta$ is the most important hyperparameter in TS. Its effect on solution quality is non-monotone:

- **$\theta$ too small (e.g., $\theta = 1$–$2$ for $n = 100$):** The tabu list is too short to prevent **cycling** — the algorithm may oscillate between a small set of solutions without making progress. Convergence is fast but to a poor local optimum.

- **$\theta$ too large (e.g., $\theta > n/2$):** Too many moves are forbidden at any given time. The **admissible neighbourhood** shrinks severely, forcing acceptance of poor moves. The algorithm loses the ability to exploit promising regions.

- **$\theta$ in the "sweet spot" (typically $\theta \approx \lfloor \sqrt{n} \rfloor$ to $n/5$):** The algorithm balances between anti-cycling memory and freedom to explore. For $n = 100$, values in the range $\theta = 7$–$20$ are commonly reported as effective.

A practical strategy is **dynamic tenure**: $\theta$ oscillates between a minimum and maximum value across iterations, preventing both pathological behaviours simultaneously.

### Comparison with Nearest-Neighbour Heuristic

On random Euclidean TSP instances, the nearest-neighbour (NN) heuristic typically produces tours **20–25% above the Held–Karp lower bound**. Tabu Search consistently reduces this gap to **5–15%**, representing a relative improvement of 10–20 percentage points over NN. The improvement is more pronounced on larger instances (higher $n$) where the NN tour has more room for 2-opt improvement.

### Comparison with GNN / Pointer Network Solvers

**Tabu Search vs. deep learning solvers** present a classic trade-off between solution quality and inference speed:

- **At inference time:** A trained Pointer Network or AM (Attention Model) can solve a $n = 100$ TSP in milliseconds on a GPU, versus seconds for TS. For batch inference, the gap is even larger.

- **On solution quality:** TS is competitive with or better than early GNN solvers on $n = 100$. However, state-of-the-art learned solvers (POMO, SGBS, etc.) with beam search can match or exceed TS quality.

- **Generalisation:** TS generalises perfectly to unseen instances (no training distribution mismatch), whereas GNN solvers may degrade on out-of-distribution instances (different $n$, different coordinate distributions).

### Approximate Optimality Gaps on TSP $n = 100$ (Random Euclidean)

The table below shows approximate **optimality gaps** (percentage above the best-known solution) for representative solvers. Values are indicative based on published literature and may vary with implementation details and instance characteristics.

| Solver | Approx. Gap (%) | Inference Time (per instance) | Requires Training Data? |
|---|---|---|---|
| **Nearest Neighbour (NN)** | 20–25% | < 1 ms | No |
| **Tabu Search (2-opt, 500 iter)** | 5–12% | 0.1–1 s | No |
| **Pointer Network (greedy)** | ~8–10% | < 10 ms (GPU) | Yes |
| **Attention Model (greedy)** | ~3–5% | < 10 ms (GPU) | Yes |
| **Attention Model (beam search)** | ~1–2% | ~100 ms (GPU) | Yes |
| **LKH-3** | < 0.5% | 1–10 s | No |
| **Concorde (exact)** | 0% | 1–60 s | No |

> **Interpretation:** TS occupies a middle ground — better than pure NN, competitive with early GNN solvers at greedy decoding, and does not require any training data. It is outperformed by modern attention-based models with beam search and by exact/LKH solvers, but remains a valuable tool when simplicity, reproducibility, and zero data cost are priorities.

---

## 11. References

1. **Glover, F. (1986).** Future paths for integer programming and links to artificial intelligence. *Computers & Operations Research*, 13(5), 533–549. https://doi.org/10.1016/0305-0548(86)90048-1

2. **Glover, F. (1989).** Tabu search — Part I. *ORSA Journal on Computing*, 1(3), 190–206. https://doi.org/10.1287/ijoc.1.3.190

3. **Glover, F. (1990).** Tabu search — Part II. *ORSA Journal on Computing*, 2(1), 4–32. https://doi.org/10.1287/ijoc.2.1.4

4. **Gendreau, M., & Potvin, J.-Y. (Eds.) (2010).** *Handbook of Metaheuristics* (2nd ed.), Chapter 2: Tabu Search. Springer. https://doi.org/10.1007/978-1-4419-1665-5

5. **Glover, F., & Laguna, M. (1997).** *Tabu Search.* Kluwer Academic Publishers. (Standard reference monograph.)

6. **Lin, S., & Kernighan, B. W. (1973).** An effective heuristic algorithm for the traveling-salesman problem. *Operations Research*, 21(2), 498–516. (Background on 2-opt and k-opt moves.)

7. **Vinyals, O., Fortunato, M., & Jaitly, N. (2015).** Pointer networks. *Advances in Neural Information Processing Systems (NeurIPS)*, 28. (Pointer Network for TSP, used for DL comparison.)

8. **Kool, W., van Hoof, H., & Welling, M. (2019).** Attention, learn to solve routing problems! *International Conference on Learning Representations (ICLR)*. (Attention Model, current GNN baseline.)